In [ ]:
import sys
from pathlib import Path

# Specify path to scripts dir for imports
scripts_dir = Path.cwd().parent / "scripts"
sys.path.insert(0, str(scripts_dir))

In [ ]:
import pandas as pd
import os

experiment_dir = "E:/Christine/100k_lymphoma/"

# Path to cohort
df_path = os.path.join(experiment_dir, "lymphoma_cohort_clean.csv")
patients_df = pd.read_csv(df_path)

In [ ]:
from helper_functions import find_files_df

# Path to WSI root dir
wsi_dir = os.path.join(experiment_dir, "slides")
suffix = ".mrxs"

slides_df, slide_paths = find_files_df(wsi_dir, suffix)
print(f"Number of {suffix} files found: ", len(slide_paths))

In [ ]:
print(slides_df.head())
print(len(slides_df), len(slide_paths))

In [ ]:
slides_df["table_id"].to_list()
slides_df = slides_df[~slides_df["table_id"].isin(['PR86_0097', 'PR86_0037', 'PR86_0059', 'PR86_0072'])]
print(len(slides_df))

In [ ]:
slides = slides_df["path"].to_list()

In [ ]:
from plate_randomization import load_microscopy_table_mapping, _format_microscopy_nr

mapping_path = "E:/Christine/100k_lymphoma/ID0089_100K_lymphoma.xlsx"

mapping = load_microscopy_table_mapping(mapping_path)
to_table_id = dict(zip(mapping["microscopy_nr"], mapping["table_id"]))

meta = patients_df.drop_duplicates(subset="rekvnr").copy()
meta["table_id"] = meta["rekvnr"].apply(_format_microscopy_nr).map(to_table_id)
print("Patients without table_id:", meta["table_id"].isna().sum())

meta["M_idx"] = 1

meta = slides_df.merge(meta, on="table_id", how="inner")   # one row per slide in `slides`
print(f"{len(slides_df)} slides, {len(meta)} with metadata")

In [ ]:
import os

# Automatically generate paths
wsi_cache = os.path.join(experiment_dir, "cache_tissue_artifact_features.pkl")
zarr_dir = os.path.join(experiment_dir, "zarr")

In [ ]:
from abmil import ABMILInference

# Run inference with trained ABMIL
abmil_path = "E:/Christine/abmil_hopt_mpp05_binary.pt"
inference_cache = os.path.join(experiment_dir, "cache_abmil_inference.pkl")
heatmap_dir = os.path.join(experiment_dir, "heatmaps")

In [ ]:
print(patients_df.head())

In [ ]:
import cv_paper_results as cv

ext = cv.from_cache(inference_cache, slides, meta, "path", "M_idx", "rekvnr")
t = cv.organ_table(ext, "group", "rekvnr")

In [ ]:
cv.table_cohort(ext, "group", "rekvnr", pos_name="Lymphoma", neg_name="Benign")

In [ ]:
cv.table_per_organ(t, "group")

In [ ]:
cv.fig_cv_roc(ext, t, "group")

In [ ]:
cv.fig_scores(ext, t, "group", pos_name="Lymphoma", neg_name="Benign")

In [ ]:
cv.fig_confusion(ext, "group", cv.ALL, pos_name="Lymphoma", neg_name="Benign")

In [ ]:
ext_dx = ext.merge(meta[["path", "diagnose"]].assign(slide=lambda m: m["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])), on="slide")
t_dx = cv.organ_table(ext_dx, "diagnose", "rekvnr")
cv.table_per_organ(t_dx, "diagnose")